# Makemore - 2

In last lecture, we implemented the bigram language model, and we implemented it both using counts and also using a super simple neural network that has a single linear layer.

The way we approached this is that we looked at only the single previous character and we predicted the distribution for the character that would go next in the sequence. And we did that by taking counts and normalizing them into probabilities so that each row sums to one.

This is all well and good if you only have one character of previous context.

The problem with this model is that the predictions from this model are not very good because you only take one character of context. And if we are to take more context into account when predicting the next character in a sequence, things quickly blow up, and the size of this table, grows exponentially with the length of the context.

Because if we only take a single character at a time, that's 27 possibilities of context. But if we take two characters in the past and try to predict the third one, suddenly the number of rows in this matrix, you can look at it that way, is 27 times 27. So there's 729 possibilities for what could have come in the context. If we take three characters as the context, suddenly we have 27*27*27, 19683, possibilities of context. So that's just too many rows of this matrix. It's way too few counts for each possibility. And the whole thing just kind of explodes and doesn't work very well.

And we're going to implement a multilayer perceptron model to predict the next character in a sequence.

And this modeling approach that we're going to adopt follows this paper, Bengio et al. 2003, A Neural Probabilities Language Model, and this paper we're going to first look at and then implement.

In this paper, they have a vocabulary of 17000 possible words, and they instead build a word-level language model. But we're going to still stick with the character, but we'll take the same modeling approach.

So what they do is basically they propose to take every one of these words, 17000 words, and they're going to associate to each word a 30-dimensional feature vector. So every word is now embedded into a 30-dimensional space. So we have 17000 points or vectors in a 30-dimensional space. And you might imagine that's very crowded. That's a lot of points for a very small space.

In the beginning, these words are initialized completely randomly. So they're spread out at random. But then we're going to tune these embeddings of these words using that propagation. So during the course of training of this neural network, these points or vectors are going to basically move around in this case. And you might imagine that, for example, words that have very similar meanings or that are indeed synonyms of each other might end up in a very similar part of the space. And conversely, words that mean very different things would go somewhere else in the space.

Now, their modeling approach otherwise is identical to ours. They are using a multilayer neural network to predict the next word, given the pervious words. And to train the neural network, they are maximizing the log likelihood of the training data, just like we did. So the modeling approach itself is identical.

Now here they have a concrete example of this intuition. Why does it work?

Basically, suppose that, for example, you are trying to predict `a dog was running in a ____`.
Now suppose that the exact phrase `a dog was running in a` has never occurred in the training data. And here you are at sort of test time later when the model is deployed somewhere and it's trying to make a sentence and it's saying `a dog was running in a ____`. And because it's never encountered this exact phrase in the training set, you're out of distribution, you don't have fundamentally any reason to suspect what might come next. But this approach actually allows you to get around that because maybe you didn't see the exact phrase, `a dog was running in a ____`, but maybe you've seen similar phrase, and maybe your network has learned that `a` and `the` are like frequently are interchangeable with each other. And so may it took the embedding for `a` and the embedding for `the` and they actually put them like nearby each other in the space. And you can transfer knowledge through that embedding and you can generalize in that way. Similarly, the network could know that cats and dogs are animals and they co-occur in lots of similar contexts and so even though you haven't seen this exact phrase or even you haven't seen exactly walking or running you can through the embedding space transfer knowledge and you can generalize to novel scenarios.

And in this example we are taking three previous words, and we are trying to predict the fourth word in a sequence. These third previous words, as I mentioned, we have a vocabulary of 17000 possible words. So everyone of these basically are the index of the incoming word. And because there are 17000 words, this is an integer between 0 to 16999. There are also a lookup table that they call `C`. This lookup table is a matrix that is 17000 by 30. And what we're doing here is we're treating this as a lookup table. And so every index is plucking out a row of this embedding matrix so that each index is converted to the 30 dimensional vector that corresponds to the embedding vector for that word. So here we have the input layer of 30 neuron for the three works, making up 90 neurons

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figure
%matplotlib inline

In [ ]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

In [ ]:
len(words)

In [ ]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.item()}
print(itos)